# 📊 RAG Pipeline Evaluation — HUSC Admissions 2025

**Tiểu luận: So sánh hiệu năng 3 Embedding Models × 3 Route Strategies**

| Embedding | Route Strategies | Metrics |
|-----------|------------------|---------|
| BGE-M3 | PaddedRAG (vector-only) | Accuracy, Recall, Hallucination |
| Harrier 0.6B | GraphRAG (vector+graph) | Groundedness, Latency p50/p95 |
| Qwen3 0.6B | Auto-route (hybrid) | Per-category breakdown |

**Runtime:** Google Colab T4 GPU • **Restart kernel → Run All** to reproduce

## 1. Setup & Prerequisites

In [4]:
import os, sys, time
print(f'Python {sys.version}')
!nvidia-smi 2>&1 | head -5
import torch
assert torch.cuda.is_available(), 'GPU not enabled. Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sat Apr 18 17:19:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
GPU: Tesla T4


In [5]:
# Load API key from Colab Secrets
from google.colab import userdata
os.environ['RAMCLOUDS_API_KEY'] = userdata.get('RAMCLOUDS_API_KEY')
_k = os.environ.get('RAMCLOUDS_API_KEY', '')
_masked = (_k[:4] + '...' + _k[-4:]) if len(_k) >= 8 else ('set-but-short' if _k else 'missing')
print(f'API key loaded: {bool(_k)} | key(masked): {_masked}')


API key loaded: True | key(masked): sk-d...yFEe


In [6]:
!git clone --depth 1 -b main https://github.com/TienDat11/husc-admission-chat-enrollment.git 2>&1 | tail -3
%cd husc-admission-chat-enrollment/rag2025
!pip install -r requirements.txt -q 2>&1 | tail -3
print('\n✅ Clone + install done')

Cloning into 'husc-admission-chat-enrollment'...
/content/husc-admission-chat-enrollment/rag2025
google-adk 1.29.0 requires PyYAML<7.0.0,>=6.0.2, but you have pyyaml 6.0.1 which is incompatible.
google-adk 1.29.0 requires requests<3.0.0,>=2.32.4, but you have requests 2.31.0 which is incompatible.
libpysal 4.14.1 requires requests>=2.32.0, but you have requests 2.31.0 which is incompatible.

✅ Clone + install done


## 2. Configuration

**IMPORTANT:** Chọn embedding model bạn muốn test. Mỗi Colab chạy 1 embedding.

In [7]:
#@title Chọn Embedding Model {display-mode: "form"}
EMBEDDING_CHOICE = 'bge' #@param ['bge', 'harrier', 'qwen']

EMBEDDING_MAP = {
    'bge':     {'model': 'BAAI/bge-m3',                    'dim': 1024},
    'harrier': {'model': 'microsoft/harrier-oss-v1-0.6b',  'dim': 1024},
    'qwen':    {'model': 'Qwen/Qwen3-Embedding-0.6B',      'dim': 1024},
}

emb_cfg = EMBEDDING_MAP[EMBEDDING_CHOICE]

# Write .env
env_content = f"""RAMCLOUDS_API_KEY={os.environ['RAMCLOUDS_API_KEY']}
RAMCLOUDS_BASE_URL=https://ramclouds.me/v1
RAMCLOUDS_MODEL=Gemma 4 31B IT
EMBEDDING_PROVIDER={EMBEDDING_CHOICE}
EMBEDDING_MODEL={emb_cfg['model']}
EMBEDDING_DIM={emb_cfg['dim']}
USE_LANCEDB=true
LANCEDB_URI=./data/lancedb
LANCEDB_TABLE=rag2025_{EMBEDDING_CHOICE}
HF_HOME=/content/hf_cache
TRANSFORMERS_CACHE=/content/hf_cache/hub
RERANKER_ENABLED=false
GUARDRAIL_ENABLED=false
ERROR_EXPOSURE_MODE=dev
"""
with open('.env', 'w') as f:
    f.write(env_content)

# Also set in os.environ for immediate use
for line in env_content.strip().split('\n'):
    if '=' in line and not line.startswith('#'):
        k, v = line.split('=', 1)
        os.environ[k] = v

print(f'✅ Config: {EMBEDDING_CHOICE} ({emb_cfg["model"]}, dim={emb_cfg["dim"]})')


✅ Config: bge (BAAI/bge-m3, dim=1024)


## 3. Patch GPU + Verify LLM

In [8]:
# Patch embedding service to use GPU (T4)
import fileinput
import json
import requests

for line in fileinput.input('src/services/embedding.py', inplace=True):
    print(line.replace('device=\"cpu\"', 'device=\"cuda\"'), end='')
print('✅ Patched embedding.py → device=cuda')

# Test LLM connectivity with structured format-error diagnostics
try:
    resp = requests.post(
        'https://ramclouds.me/v1/chat/completions',
        headers={'Authorization': f'Bearer {os.environ["RAMCLOUDS_API_KEY"]}'},
        json={
            'model': os.environ['RAMCLOUDS_MODEL'],
            'messages': [{'role': 'user', 'content': 'ping'}],
            'max_tokens': 5,
        },
        timeout=15,
    )

    content_type = resp.headers.get('content-type', 'unknown')
    body_preview = (resp.text or '')[:240]

    if resp.status_code == 200:
        try:
            data = resp.json()
            msg = data.get('choices', [{}])[0].get('message', {}).get('content', '')
            print(f'✅ LLM OK: {msg}')
        except Exception as e:
            print('⚠️ LLM_FORMAT_ERROR')
            print(json.dumps({
                'stage': 'llm_ping_json_parse',
                'error_type': type(e).__name__,
                'error_message': str(e),
                'status_code': resp.status_code,
                'content_type': content_type,
                'body_preview': body_preview,
                'model': os.environ.get('RAMCLOUDS_MODEL'),
            }, ensure_ascii=False, indent=2))
    else:
        print('❌ LLM_HTTP_ERROR')
        print(json.dumps({
            'stage': 'llm_ping_http',
            'status_code': resp.status_code,
            'content_type': content_type,
            'body_preview': body_preview,
            'model': os.environ.get('RAMCLOUDS_MODEL'),
        }, ensure_ascii=False, indent=2))

except Exception as e:
    print('❌ LLM_CONNECTION_ERROR')
    print(json.dumps({
        'stage': 'llm_ping_request',
        'error_type': type(e).__name__,
        'error_message': str(e),
        'model': os.environ.get('RAMCLOUDS_MODEL'),
    }, ensure_ascii=False, indent=2))


✅ Patched embedding.py → device=cuda
✅ LLM OK: pong


## 4. Data Ingest → LanceDB + Knowledge Graph

In [ ]:
import json
import subprocess

# Check where the scripts actually are
import os
if not os.path.exists('/content/husc-admission-chat-enrollment'):
    print("Repo not found. Re-cloning...")
    !git clone --depth 1 -b main https://github.com/TienDat11/husc-admission-chat-enrollment.git /content/husc-admission-chat-enrollment

# Find the correct base path
possible_paths = [
    '/content/husc-admission-chat-enrollment/rag2025',
    '/content/husc-admission-chat-enrollment',
    '/content/rag2025'
]

BASE_PATH = None
for p in possible_paths:
    if os.path.exists(os.path.join(p, 'scripts/ingest_lancedb.py')):
        BASE_PATH = p
        break

if BASE_PATH:
    %cd {BASE_PATH}
    print(f"Using BASE_PATH: {BASE_PATH}")
else:
    print("❌ Error: Could not find scripts directory!")

# Data check
!ls -la data/chunked/ 2>/dev/null || echo "Data dir missing"
!echo "---"

def run_step(step_name, cmd):
    print(f'\n>>> {step_name}')
    proc = subprocess.run(cmd, capture_output=True, text=True, env=os.environ.copy())
    if proc.stdout: print(proc.stdout)
    if proc.stderr: print(proc.stderr)
    if proc.returncode != 0:
        print('❌ STEP_FAILED')
        raise RuntimeError(f'{step_name} failed')

# Execute steps
run_step('Step 1: LanceDB Ingest', ['python', 'scripts/ingest_lancedb.py'])
os.environ['HF_HOME'] = '/content/hf_cache'
run_step('Step 2: Build Knowledge Graph', ['python', 'scripts/build_graph.py'])
run_step('Step 3: Verification', ['python', 'scripts/preflight_check.py'])

/content/husc-admission-chat-enrollment/rag2025
Using BASE_PATH: /content/husc-admission-chat-enrollment/rag2025
total 368
drwxr-xr-x 2 root root  4096 Apr 18 17:19 .
drwxr-xr-x 6 root root  4096 Apr 18 17:19 ..
-rw-r--r-- 1 root root 60054 Apr 18 17:19 chunked_10_enhanced.jsonl
-rw-r--r-- 1 root root 52830 Apr 18 17:19 chunked_10.jsonl
-rw-r--r-- 1 root root 20083 Apr 18 17:19 chunked_11.jsonl
-rw-r--r-- 1 root root 43984 Apr 18 17:19 chunked_1.jsonl
-rw-r--r-- 1 root root  3933 Apr 18 17:19 chunked_2.jsonl
-rw-r--r-- 1 root root 87110 Apr 18 17:19 chunked_3.jsonl
-rw-r--r-- 1 root root  4980 Apr 18 17:19 chunked_4.jsonl
-rw-r--r-- 1 root root 21710 Apr 18 17:19 chunked_5.jsonl
-rw-r--r-- 1 root root  6070 Apr 18 17:19 chunked_6.jsonl
-rw-r--r-- 1 root root     0 Apr 18 17:19 chunked_7.jsonl
-rw-r--r-- 1 root root 34526 Apr 18 17:19 chunked_8.jsonl
-rw-r--r-- 1 root root  8400 Apr 18 17:19 chunked_9.jsonl
-rw-r--r-- 1 root root  3564 Apr 18 17:19 chunked_test.jsonl
---

>>> Step 1: La

## 5. Start Pipeline Server

In [ ]:
import subprocess, time

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'src.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(20)

try:
    r = requests.get('http://127.0.0.1:8000/health', timeout=10)
    print(f'✅ Server running: {r.status_code}')
except Exception as e:
    print(f'❌ Server failed: {e}')
    !tail -40 /content/server.log

## 6. Full Evaluation — 3 Routes × 50 Questions

Tests: `padded_rag` (vector-only), `graph_rag` (vector+graph), `auto` (hybrid router)

In [ ]:
import json, time, requests, os
sys.path.insert(0, '/content/husc-admission-chat-enrollment/packages/rag-chatbot-husc/src')

from notebooks.eval_core import (
    load_test_questions, normalize_pipeline_output,
    exact_correctness, retrieval_recall_proxy, hallucination_flag,
    GROUNDING_THRESHOLD,
)

EMB = os.environ['EMBEDDING_PROVIDER']
BASE_URL = 'http://127.0.0.1:8000'
ROUTES = ['padded_rag', 'graph_rag', None]  # None = auto-route
ROUTE_NAMES = {None: 'auto', 'padded_rag': 'padded_rag', 'graph_rag': 'graph_rag'}

rows, used_path = load_test_questions(
    'results/test_questions.json',
    'backup_mail_package_2026/python_project/rag2025/results/test_questions.json',
)
print(f'Loaded {len(rows)} questions from {used_path}')

os.makedirs('results/colab_eval', exist_ok=True)

all_results = []

for route in ROUTES:
    route_name = ROUTE_NAMES[route]
    print(f'\n{"="*60}')
    print(f'>>> Route: {route_name} | Embedding: {EMB}')
    print(f'{"="*60}')

    for i, item in enumerate(rows):
        q = item['question']
        start = time.perf_counter()
        try:
            payload = {'query': q, 'top_k': 5}
            if route is not None:
                payload['force_route'] = route

            resp = requests.post(f'{BASE_URL}/v2/query', json=payload, timeout=120)
            resp.raise_for_status()
            raw = resp.json()
            elapsed = time.perf_counter() - start
            norm = normalize_pipeline_output(raw, mode='v2')

            score = exact_correctness(norm['answer'], item.get('ground_truth_answer', ''))
            recall = retrieval_recall_proxy(
                norm.get('source_ids', []),
                item.get('ground_truth_source_chunks', []),
            )
            halluc = hallucination_flag(norm.get('groundedness_score', 0.0), GROUNDING_THRESHOLD)

            entry = {
                'idx': i, 'question': q,
                'answer': norm['answer'],
                'gt': item.get('ground_truth_answer', ''),
                'score_exact': score, 'recall': recall, 'hallucination': halluc,
                'route_requested': route_name,
                'route_actual': norm.get('route', ''),
                'latency_ms': round(elapsed * 1000, 1),
                'groundedness': norm.get('groundedness_score', 0.0),
                'embedding': EMB,
                'category': item.get('category', ''),
            }
        except Exception as e:
            entry = {
                'idx': i, 'question': q, 'error': str(e),
                'route_requested': route_name, 'embedding': EMB,
                'category': item.get('category', ''),
                'score_exact': 0, 'recall': 0, 'hallucination': 1,
                'latency_ms': 0, 'groundedness': 0.0,
            }

        all_results.append(entry)

        if (i + 1) % 10 == 0:
            ok = [r for r in all_results if r.get('route_requested') == route_name and 'error' not in r]
            acc = sum(r['score_exact'] for r in ok) / len(ok) if ok else 0
            print(f'  [{i+1}/{len(rows)}] acc={acc:.2%}')

# Save full log
log_path = f'results/colab_eval/query_log_{EMB}_full.jsonl'
with open(log_path, 'w', encoding='utf-8') as f:
    for r in all_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f'\n✅ Done: {len(all_results)} total evaluations → {log_path}')

## 7. Summary Statistics

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame(all_results)
df_ok = df[~df['error'].notna()] if 'error' in df.columns else df
# Filter out errors
df_ok = df[df.apply(lambda r: 'error' not in r or pd.isna(r.get('error')), axis=1)].copy()

# Per-route summary
summary = df.groupby('route_requested').agg(
    count=('score_exact', 'count'),
    accuracy=('score_exact', 'mean'),
    recall=('recall', 'mean'),
    hallucination_rate=('hallucination', 'mean'),
    groundedness=('groundedness', 'mean'),
    latency_p50=('latency_ms', 'median'),
    latency_p95=('latency_ms', lambda x: np.percentile(x, 95)),
).round(4)

print(f'\n=== {EMB.upper()} — Route Comparison ===')
print(summary.to_string())

# Per-category × route
cat_summary = df.groupby(['category', 'route_requested']).agg(
    count=('score_exact', 'count'),
    accuracy=('score_exact', 'mean'),
    recall=('recall', 'mean'),
    hallucination_rate=('hallucination', 'mean'),
).round(4)

print(f'\n=== Per-Category × Route ===')
print(cat_summary.to_string())

# Save
summary.to_csv(f'results/colab_eval/summary_{EMB}.csv')
cat_summary.to_csv(f'results/colab_eval/category_summary_{EMB}.csv')

## 8. Charts for Thesis Report 📊

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

# ====== Chart 1: Accuracy by Route ======
fig, ax = plt.subplots(figsize=(8, 5))
routes = summary.index.tolist()
acc_vals = summary['accuracy'].values
colors = ['#2196F3', '#4CAF50', '#FF9800']

bars = ax.bar(routes, acc_vals, color=colors[:len(routes)], edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', va='bottom', fontweight='bold')

ax.set_title(f'Accuracy by Route Strategy — {EMB.upper()}', fontweight='bold', fontsize=14)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Target 80%')
ax.legend()
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_accuracy_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 2: Latency Distribution (Box Plot) ======
fig, ax = plt.subplots(figsize=(8, 5))

route_groups = [df[df['route_requested'] == r]['latency_ms'].dropna().values for r in routes]
bp = ax.boxplot(route_groups, labels=routes, patch_artist=True, showfliers=True)

for patch, color in zip(bp['boxes'], colors[:len(routes)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title(f'Latency Distribution by Route — {EMB.upper()}', fontweight='bold', fontsize=14)
ax.set_ylabel('Latency (ms)')
ax.axhline(y=summary['latency_p95'].max(), color='red', linestyle='--', alpha=0.5, label=f'P95 max: {summary["latency_p95"].max():.0f}ms')
ax.legend()
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_latency_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 3: Radar Chart — Multi-metric Comparison ======
from math import pi

metrics = ['accuracy', 'recall', 'groundedness']
# Invert hallucination (1 - rate) so higher = better on radar
metric_labels = ['Accuracy', 'Recall', 'Groundedness', '1 - Hallucination']

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

N = len(metric_labels)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

for idx, route in enumerate(routes):
    row = summary.loc[route]
    values = [
        row['accuracy'],
        row['recall'],
        row['groundedness'],
        1 - row['hallucination_rate'],
    ]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=route, color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title(f'Multi-metric Radar — {EMB.upper()}', fontweight='bold', fontsize=14, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_radar_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 4: Category Breakdown (Grouped Bar) ======
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

categories = df['category'].unique()
x = np.arange(len(categories))
width = 0.25

for metric_idx, (metric, title) in enumerate([('score_exact', 'Accuracy'), ('recall', 'Recall')]):
    ax = axes[metric_idx]
    for r_idx, route in enumerate(routes):
        vals = [df[(df['category'] == cat) & (df['route_requested'] == route)][metric].mean()
                for cat in categories]
        ax.bar(x + r_idx * width, vals, width, label=route, color=colors[r_idx], alpha=0.8)

    ax.set_title(f'{title} by Category — {EMB.upper()}', fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(categories)
    ax.set_ylabel(title)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_category_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 5: Heatmap — Category × Route Accuracy ======
import seaborn as sns

pivot = df.pivot_table(values='score_exact', index='category', columns='route_requested', aggfunc='mean')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=1, ax=ax, cbar_kws={'label': 'Accuracy'})
ax.set_title(f'Accuracy Heatmap: Category × Route — {EMB.upper()}', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_heatmap_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====== Chart 6: Error Analysis — Stacked Bar ======
fig, ax = plt.subplots(figsize=(10, 5))

error_types = []
for _, row in df.iterrows():
    if row.get('score_exact', 0) == 0:
        if row.get('hallucination', 0) == 1:
            error_types.append('hallucination')
        elif row.get('recall', 0) == 0:
            error_types.append('retrieval_miss')
        else:
            error_types.append('reasoning_error')
    else:
        error_types.append('correct')

df['error_type'] = error_types

error_pivot = df.groupby(['route_requested', 'error_type']).size().unstack(fill_value=0)
error_colors = {'correct': '#4CAF50', 'hallucination': '#F44336', 'retrieval_miss': '#FF9800', 'reasoning_error': '#9C27B0'}
col_order = [c for c in ['correct', 'retrieval_miss', 'hallucination', 'reasoning_error'] if c in error_pivot.columns]
error_pivot[col_order].plot(kind='bar', stacked=True, ax=ax,
                            color=[error_colors[c] for c in col_order], edgecolor='black', linewidth=0.3)

ax.set_title(f'Error Type Distribution by Route — {EMB.upper()}', fontweight='bold', fontsize=14)
ax.set_ylabel('Count')
ax.set_xlabel('Route')
ax.legend(title='Type', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'results/colab_eval/chart_errors_{EMB}.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Top Failed Questions (for Analysis)

In [ ]:
# Show worst questions across all routes
fails = df[df['score_exact'] == 0].copy()
if len(fails) > 0:
    # Count failures per question
    fail_counts = fails.groupby('question').agg(
        fail_routes=('route_requested', lambda x: ', '.join(x)),
        avg_groundedness=('groundedness', 'mean'),
        total_fails=('score_exact', 'count'),
    ).sort_values('total_fails', ascending=False).head(10)

    print(f'=== Top 10 Most Failed Questions ({EMB.upper()}) ===')
    for i, (q, row) in enumerate(fail_counts.iterrows(), 1):
        print(f'\n{i}. [{row["total_fails"]}/3 routes failed] Q: {q[:80]}...')
        print(f'   Failed on: {row["fail_routes"]}')
        print(f'   Avg groundedness: {row["avg_groundedness"]:.3f}')
else:
    print('No failures! All questions answered correctly.')

## 10. Generate Thesis Report

In [ ]:
report_lines = []
report_lines.append(f'# Báo Cáo Đánh Giá RAG Pipeline — {EMB.upper()}')
report_lines.append(f'\n**Ngày:** {time.strftime("%Y-%m-%d %H:%M")}')
report_lines.append(f'**Embedding:** {emb_cfg["model"]} (dim={emb_cfg["dim"]})')
report_lines.append(f'**Số câu hỏi:** {len(rows)}')
report_lines.append(f'**Route strategies:** padded_rag, graph_rag, auto')

report_lines.append('\n## 1. Tổng quan kết quả')
report_lines.append('\n| Route | N | Accuracy | Recall | Hallucination | Groundedness | Latency p50 | Latency p95 |')
report_lines.append('|-------|---|----------|--------|---------------|-------------|-------------|-------------|')
for route in routes:
    route_name = ROUTE_NAMES[route]
    row = summary.loc[route_name]
    report_lines.append(
        f'| {route_name} | {int(row["count"])} '
        f'| {row["accuracy"]:.2%} | {row["recall"]:.2%} '
        f'| {row["hallucination_rate"]:.2%} | {row["groundedness"]:.3f} '
        f'| {row["latency_p50"]:.0f}ms | {row["latency_p95"]:.0f}ms |'
    )

# Best route
best_route = summary['accuracy'].idxmax()
report_lines.append(f'\n**Route tốt nhất theo accuracy:** `{best_route}` ({summary.loc[best_route, "accuracy"]:.2%})')

report_lines.append('\n## 2. Phân tích theo Category')
report_lines.append('\n| Category | Route | N | Accuracy | Recall | Hallucination |')
report_lines.append('|----------|-------|---|----------|--------|---------------|')
for (cat, route), row in cat_summary.iterrows():
    report_lines.append(
        f'| {cat} | {route} | {int(row["count"])} '
        f'| {row["accuracy"]:.2%} | {row["recall"]:.2%} '
        f'| {row["hallucination_rate"]:.2%} |'
    )

report_lines.append('\n## 3. Phân tích lỗi')
if len(fails) > 0:
    error_summary = df['error_type'].value_counts()
    report_lines.append('\n| Loại lỗi | Số lượng | Tỷ lệ |')
    report_lines.append('|----------|----------|-------|')
    for etype, count in error_summary.items():
        report_lines.append(f'| {etype} | {count} | {count/len(df):.2%} |')

report_lines.append('\n## 4. Charts')
report_lines.append(f'\n![Accuracy](chart_accuracy_{EMB}.png)')
report_lines.append(f'![Latency](chart_latency_{EMB}.png)')
report_lines.append(f'![Radar](chart_radar_{EMB}.png)')
report_lines.append(f'![Category](chart_category_{EMB}.png)')
report_lines.append(f'![Heatmap](chart_heatmap_{EMB}.png)')
report_lines.append(f'![Errors](chart_errors_{EMB}.png)')

report_lines.append('\n## 5. Nhận xét & Khuyến nghị')
report_lines.append('\n### Điểm mạnh')
if summary['accuracy'].max() >= 0.8:
    report_lines.append(f'- Route `{best_route}` đạt accuracy ≥ 80% target')
if summary['hallucination_rate'].min() < 0.15:
    report_lines.append(f'- Hallucination rate thấp ({summary["hallucination_rate"].min():.2%})')

report_lines.append('\n### Điểm yếu cần cải thiện')
worst_route = summary['accuracy'].idxmin()
if summary.loc[worst_route, 'accuracy'] < 0.7:
    report_lines.append(f'- Route `{worst_route}` accuracy thấp ({summary.loc[worst_route, "accuracy"]:.2%})')
if summary['latency_p95'].max() > 5000:
    report_lines.append(f'- Latency p95 cao ({summary["latency_p95"].max():.0f}ms) — cần optimize')

report_lines.append('\n### Roadmap cải tiến')
report_lines.append('1. **Quick wins (1-3 ngày):** Tune top_k, adjust groundedness threshold')
report_lines.append('2. **Medium (1-2 tuần):** Thêm reranker, query rewriting')
report_lines.append('3. **Long-term (2-6 tuần):** Hard negative mining, graph reasoning optimization')

# Write report
report_text = '\n'.join(report_lines)
with open(f'results/colab_eval/thesis_report_{EMB}.md', 'w', encoding='utf-8') as f:
    f.write(report_text)

print(report_text)

## 11. Download Results

In [ ]:
from google.colab import files
import glob

print(f'=== Files to download ({EMB.upper()}) ===')
all_files = glob.glob('results/colab_eval/*')
for f in sorted(all_files):
    print(f'  {f}')

# Download all
for f in all_files:
    try:
        files.download(f)
    except Exception as e:
        print(f'  Skip {f}: {e}')